# Recommendation demo

For one user, compare the implicit top five, explicit top five, and a combined top five. The combined ranking takes the explicit top 1,000 books and sorts them by `explicit_score * implicit_probability`. Books already seen by the user are excluded.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from bookrec.data import ITEM_COLUMN, USER_COLUMN, load_dataset
from bookrec.explicit.hyperparameters import DEFAULT_HYPERPARAMETERS as EXPLICIT_DEFAULTS
from bookrec.explicit.model import ExplicitRecommenderMLP
from bookrec.implicit.hyperparameters import DEFAULT_HYPERPARAMETERS as IMPLICIT_DEFAULTS
from bookrec.implicit.model import create_implicit_model

# Change these two values to select the user and implicit model.
USER_ID = 11676
IMPLICIT_MODEL_NAME = "mlp"  # "mlp" or "neumf"

TOP_K = 5
EXPLICIT_POOL_SIZE = 1_000
BATCH_SIZE = 16_384
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ARTIFACTS = Path("artifacts")
print(f"Device: {DEVICE}")

In [ ]:
ratings = load_dataset("Ratings.csv")
books = load_dataset("Books.csv")
ratings[ITEM_COLUMN] = ratings[ITEM_COLUMN].astype(str)
books[ITEM_COLUMN] = books[ITEM_COLUMN].astype(str)

metadata = (
    books[[
        ITEM_COLUMN,
        "Book-Title",
        "Book-Author",
        "Year-Of-Publication",
        "Publisher",
        "Image-URL-M",
    ]]
    .drop_duplicates(ITEM_COLUMN)
)
metadata_isbns = set(metadata[ITEM_COLUMN])

implicit_checkpoint = torch.load(
    ARTIFACTS / "implicit" / IMPLICIT_MODEL_NAME / "model_with_mappings.pt",
    map_location="cpu",
    weights_only=False,
)
implicit_users = implicit_checkpoint["user_to_index"]
implicit_items = implicit_checkpoint["item_to_index"]
implicit_hparams = implicit_checkpoint.get("hyperparameters", IMPLICIT_DEFAULTS)
implicit_model = create_implicit_model(
    IMPLICIT_MODEL_NAME,
    len(implicit_users),
    len(implicit_items),
    implicit_hparams,
).to(DEVICE)
implicit_model.load_state_dict(implicit_checkpoint["model_state_dict"])
implicit_model.eval()

explicit_checkpoint = torch.load(
    ARTIFACTS / "explicit" / "mlp" / "model_with_mappings.pt",
    map_location="cpu",
    weights_only=False,
)
explicit_users = explicit_checkpoint["user_to_index"]
explicit_items = explicit_checkpoint["item_to_index"]
explicit_hparams = explicit_checkpoint.get("hyperparameters", EXPLICIT_DEFAULTS)
explicit_model = ExplicitRecommenderMLP(
    len(explicit_users),
    len(explicit_items),
    embedding_dim=explicit_hparams["embedding_dim"],
    hidden_dims=tuple(explicit_hparams["hidden_dims"]),
    dropout=explicit_hparams["dropout"],
).to(DEVICE)
explicit_model.load_state_dict(explicit_checkpoint["model_state_dict"])
explicit_model.eval()

if USER_ID not in implicit_users or USER_ID not in explicit_users:
    raise ValueError(f"User-ID {USER_ID} must be known to both models")

In [ ]:
def score_items(model, user_index, item_to_index):
    index_to_isbn = [None] * len(item_to_index)
    for isbn, item_index in item_to_index.items():
        index_to_isbn[item_index] = str(isbn)

    batches = []
    with torch.inference_mode():
        for start in range(0, len(index_to_isbn), BATCH_SIZE):
            stop = min(start + BATCH_SIZE, len(index_to_isbn))
            items = torch.arange(start, stop, device=DEVICE)
            users = torch.full_like(items, user_index)
            batches.append(model(users, items).cpu())

    return pd.DataFrame({
        ITEM_COLUMN: index_to_isbn,
        "score": torch.cat(batches).numpy(),
    })

In [ ]:
from scipy.special import expit

seen_isbns = set(
    ratings.loc[ratings[USER_COLUMN] == USER_ID, ITEM_COLUMN]
)

implicit_scores = score_items(
    implicit_model, implicit_users[USER_ID], implicit_items
).rename(columns={"score": "implicit_logit"})
implicit_logits = implicit_scores["implicit_logit"].to_numpy()
implicit_scores["implicit_probability"] = expit(implicit_logits)
implicit_scores = implicit_scores[
    implicit_scores[ITEM_COLUMN].isin(metadata_isbns)
    & ~implicit_scores[ITEM_COLUMN].isin(seen_isbns)
]

explicit_scores = score_items(
    explicit_model, explicit_users[USER_ID], explicit_items
).rename(columns={"score": "explicit_score"})
explicit_scores = explicit_scores[
    explicit_scores[ITEM_COLUMN].isin(metadata_isbns)
    & ~explicit_scores[ITEM_COLUMN].isin(seen_isbns)
]

implicit_top = (
    implicit_scores
    .nlargest(TOP_K, "implicit_probability")
    .merge(metadata, on=ITEM_COLUMN)
)
explicit_top = (
    explicit_scores
    .nlargest(TOP_K, "explicit_score")
    .merge(metadata, on=ITEM_COLUMN)
)

combined_pool = (
    explicit_scores
    .merge(
        implicit_scores[[ITEM_COLUMN, "implicit_probability"]],
        on=ITEM_COLUMN,
    )
    .nlargest(EXPLICIT_POOL_SIZE, "explicit_score")
)
combined_pool["combined_score"] = (
    combined_pool["explicit_score"]
    * combined_pool["implicit_probability"]
)
combined_top = (
    combined_pool
    .nlargest(TOP_K, "combined_score")
    .merge(metadata, on=ITEM_COLUMN)
)

In [ ]:
metadata_columns = [
    "Book-Title",
    "Book-Author",
    "Year-Of-Publication",
    "Publisher",
    ITEM_COLUMN,
    "Image-URL-M",
]

print(f"User-ID {USER_ID} — implicit model: {IMPLICIT_MODEL_NAME}")
print("Top 5 — implicit")
display(implicit_top[[*metadata_columns, "implicit_probability"]])

print("Top 5 — explicit")
display(explicit_top[[*metadata_columns, "explicit_score"]])

print("Top 5 — combined")
display(combined_top[[
    *metadata_columns,
    "explicit_score",
    "implicit_probability",
    "combined_score",
]])